# Mini-lab: do vetor adversarial à manipulação de payload

Neste laboratório, você executará a avaliação sem perturbação de um **NIDS**, gerará ataques no espaço de características e comparará os resultados com três cenários de manipulação de payload no espaço do problema.

Selecione **Ambiente de execução > Executar tudo**. Examine as etapas 1–4 e registre três observações na etapa 5.

### Glossário

| Termo | Significado neste laboratório |
|---|---|
| **NIDS** | sistema de detecção de intrusão em rede |
| **DoS** | negação de serviço; classe de ataque avaliada |
| **FGM minimal** | Fast Gradient Method com busca da menor perturbação até `ε` |
| **PGD** | Projected Gradient Descent, ataque iterativo limitado por `ε` |
| **ASR** | taxa de sucesso do ataque no espaço de características |
| **Recall** | proporção dos fluxos DoS detectados: `TP / (TP + FN)` |
| **FNR** | proporção dos fluxos DoS não detectados: `FN / (TP + FN)` |

As manipulações `+10%`, `+50%` e `+100%` de **payload** e as respectivas
reextrações de características são pré-computadas. A manipulação foi
operacionalizada por `IP.len`.


## 1. Preparação e configuração

Os valores abaixo reproduzem o roteiro principal. Para `GRUPO_CARACTERISTICAS`, escolha `"volume"`, `"tempo"` ou `"todas"`.

**Pipeline NIDS:** TRÁFEGO/PCAP -> EXTRAÇÃO -> [**CARACTERÍSTICAS**] -> [**SCALER**] -> [**MLP**] -> DECISÃO -> AVALIAÇÃO

> **Foco desta seção:** O código carrega os recortes de fluxos já convertidos em características, o scaler e a MLP pré-treinada.


In [ ]:
# Parâmetros do experimento
N_AMOSTRAS = 4788
LIMIAR_DECISAO = 0.30
EPSILON = 0.005
SEMENTE = 20260901
GRUPO_CARACTERISTICAS = 'volume'
ITERACOES_PGD = 30

from pathlib import Path
import subprocess
import sys

REPOSITORIO = "https://github.com/espindolaallan/sbseg-2026-minicurso-nids-adversarial.git"
ROOT = Path("/content/sbseg-2026-minicurso-nids-adversarial")
if not ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", REPOSITORIO, str(ROOT)],
        check=True,
    )
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "adversarial-robustness-toolbox==1.20.1",
    ],
    check=True,
)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display

from aml_nids_lab.ataques import fluxos_elegiveis
from aml_nids_lab.avaliacao import metricas_classificacao, metricas_payload
from aml_nids_lab.caracteristicas import carregar_caracteristicas
from aml_nids_lab.configuracao import CENARIOS_PAYLOAD, CONJUNTOS, criar_parametros
from aml_nids_lab.dados import carregar_dados
from aml_nids_lab.detector import carregar_detector, inferir
from aml_nids_lab.experimento import (
    ResultadoBaseline,
    avaliar_ataques,
    executar_ataques,
    montar_avaliacao_payload,
    preparar_pipeline,
    selecionar_ataques,
)

parametros = criar_parametros(
    n_amostras=N_AMOSTRAS,
    limiar=LIMIAR_DECISAO,
    epsilon=EPSILON,
    semente=SEMENTE,
    grupo_caracteristicas=GRUPO_CARACTERISTICAS,
    iteracoes_pgd=ITERACOES_PGD,
)
caracteristicas = carregar_caracteristicas(ROOT)
dados = {
    nome: carregar_dados(ROOT, nome, caracteristicas=caracteristicas)
    for nome in CONJUNTOS
}
modelo = carregar_detector(ROOT)
pipeline = preparar_pipeline(parametros, caracteristicas, dados, modelo)

def pct(value):
    return "—" if value is None else f"{100 * value:.2f}%".replace(".", ",")

n_features = len(caracteristicas.nomes)
n_manipulaveis = int(caracteristicas.mascara(parametros.grupo_caracteristicas).sum())
display(pd.DataFrame([
    {"Parâmetro": "Semente", "Valor": SEMENTE},
    {"Parâmetro": "Amostra selecionada / denominador", "Valor": N_AMOSTRAS},
    {"Parâmetro": "Limiar do NIDS", "Valor": LIMIAR_DECISAO},
    {"Parâmetro": "ε e norma", "Valor": f"{EPSILON} / L∞"},
    {"Parâmetro": "Características manipuláveis", "Valor": f"{n_manipulaveis} de {n_features} ({parametros.grupo_caracteristicas})"},
    {"Parâmetro": "Ataques", "Valor": f"FGM minimal e PGD ({ITERACOES_PGD} iterações)"},
    {"Parâmetro": "Dispositivo", "Valor": "CPU"},
]).style.hide(axis="index"))


## 2. Avaliação sem perturbação e população elegível

A avaliação sem perturbação usa o conjunto limpo. Sua matriz de confusão mostra `TP`, `TN`, `FP` e `FN`. Para os ataques no espaço de características, a população elegível contém apenas fluxos DoS corretamente detectados antes das transformações; a amostra determinística dessa população será `E_N`.


**Pipeline NIDS:** TRÁFEGO/PCAP -> EXTRAÇÃO -> CARACTERÍSTICAS -> [**SCALER**] -> [**MLP**] -> [**DECISÃO**] -> [**AVALIAÇÃO**]

> **Foco desta seção:** As características fornecidas passam pelo scaler, pela MLP e pelo limiar para produzir a avaliação de referência.


In [ ]:
dados_limpos = dados["clean_day17"]
vetores_limpos = caracteristicas.normalizar(dados_limpos.x)
inferencia_limpa = inferir(modelo, vetores_limpos, LIMIAR_DECISAO)
rotulos_limpos = tuple(int(valor) for valor in dados_limpos.y)
predicoes_limpas = tuple(int(valor) for valor in inferencia_limpa.rotulos)
metricas_limpas = metricas_classificacao(rotulos_limpos, predicoes_limpas)
ids_elegiveis = fluxos_elegiveis(
    dados_limpos.ids,
    rotulos_limpos,
    predicoes_limpas,
)
resultado_baseline = ResultadoBaseline(
    metricas_limpas,
    tuple(str(identificador) for identificador in ids_elegiveis),
    predicoes_limpas,
)
baseline = resultado_baseline.metricas.to_dict()
elegiveis = len(resultado_baseline.ids_elegiveis)

display(pd.DataFrame([{
    "Fluxos": baseline["population"],
    "TP": baseline["tp"],
    "TN": baseline["tn"],
    "FP": baseline["fp"],
    "FN": baseline["fn"],
    "Recall": pct(baseline["recall"]),
    "FNR": pct(baseline["fnr"]),
    "Elegíveis": elegiveis,
}]).style.hide(axis="index"))

display(Markdown(
    f"**Leitura:** o NIDS detectou **{baseline['tp']} de {baseline['tp'] + baseline['fn']}** "
    f"fluxos DoS no conjunto limpo (Recall **{pct(baseline['recall'])}**). "
    f"Há **{elegiveis}** verdadeiros positivos disponíveis para formar `E_N`."
))


## 3. FGM minimal e PGD no espaço de características

Os dois ataques intervêm no vetor normalizado produzido pelo scaler. Eles usam a mesma amostra `E_N`, o grupo de características e o limite `ε` selecionados na etapa 1.

`ASR = evasões / |E_N|`

Uma evasão ocorre quando um fluxo DoS detectado antes da transformação passa a ser classificado como normal. “Evasão coerente” também exige aprovação nas verificações tabulares; isso ainda não demonstra que o vetor possa ser realizado como tráfego de rede.


**Pipeline NIDS:** TRÁFEGO/PCAP -> EXTRAÇÃO -> CARACTERÍSTICAS -> SCALER -> [**FGM MINIMAL / PGD**] -> MLP -> DECISÃO -> AVALIAÇÃO

> **Foco desta seção:** FGM minimal e PGD atuam nos vetores pós-scaler preparados em [0, 1], antes da MLP; as saídas seguem para decisão e avaliação.


In [ ]:
selecao = selecionar_ataques(pipeline, resultado_baseline, vetores_limpos)
ataques_executados = executar_ataques(pipeline, selecao)
avaliacao_ataques = avaliar_ataques(pipeline, selecao, ataques_executados)

display(Markdown(
    f"**Amostra comum:** {selecao.avaliacao.efetiva} casos enviados aos dois "
    "geradores e usados no denominador."
))

feature_rows = []
for method, item in (("FGM", avaliacao_ataques.fgm), ("PGD", avaliacao_ataques.pgd)):
    metrics = item.metricas.to_dict()
    disponivel = (
        item.status == "available"
        and len(item.execucao.ids_saida) > 0
    )
    cost = {} if item.custo is None else item.custo.to_dict()
    l0_median = cost.get("changed_features_median")
    linf_median = cost.get("linf_median")
    cost_text = (
        "—"
        if l0_median is None
        else f"{l0_median:.0f} / {linf_median:.5f}".replace(".", ",")
    )
    feature_rows.append({
        "Técnica": "FGM minimal" if method == "FGM" else "PGD",
        "Status": "Disponível" if disponivel else "Indisponível",
        "N": metrics["eligible"],
        "Evasões": metrics["evasions"] if disponivel else "—",
        "ASR": pct(metrics["asr"]) if disponivel else "—",
        "Evasões coerentes": metrics["coherent_evasions"] if disponivel else "—",
        "Magnitude mediana L0 / L∞": cost_text,
        "Tempo": f"{item.execucao.segundos:.2f} s",
    })

display(pd.DataFrame(feature_rows).style.hide(axis="index"))
display(Markdown(
    f"**Leitura da magnitude:** `L0` e `L∞` resumem todas as "
    f"{len(avaliacao_ataques.fgm.execucao.ids_saida)} saídas de FGM minimal e "
    f"{len(avaliacao_ataques.pgd.execucao.ids_saida)} saídas de PGD. "
    "O tempo de execução é medido separadamente."
))

feature_lines = []
for row in feature_rows:
    if row["Status"] == "Indisponível":
        feature_lines.append(
            f"- **{row['Técnica']}**: indisponível; o ataque não produziu saídas avaliáveis."
        )
    else:
        feature_lines.append(
            f"- **{row['Técnica']}**: {row['Evasões']} evasões em N={row['N']} "
            f"(ASR **{row['ASR']}**); {row['Evasões coerentes']} passaram "
            "todas as verificações tabulares."
        )
display(Markdown("\n".join(feature_lines)))


## 4. Manipulação de payload no espaço do problema

Os níveis de payload +10%, +50% e +100% foram aplicados antes da reextração. Um nível maior não precisa produzir Recall menor ou FNR maior.

![Manipulação no espaço do problema antes da extração de características](https://raw.githubusercontent.com/espindolaallan/sbseg-2026-minicurso-nids-adversarial/main/assets/d_mood_problem_space_pipeline.png)

*A manipulação de payload ($\delta^{vol}$) atua nos pacotes antes da extração de características. Fonte: [D-MOOD](https://github.com/espindolaallan/d-mood).*

Como os conjuntos não mantêm pareamento individual com o conjunto limpo, reportamos **TP**, **FN**, **Recall** e **FNR** — não ASR.

**Pipeline NIDS:** [**TRÁFEGO/PCAP**] -> EXTRAÇÃO -> CARACTERÍSTICAS -> SCALER -> MLP -> DECISÃO -> AVALIAÇÃO

> **Foco desta seção:** A intervenção no tráfego e a extração foram realizadas previamente; nesta etapa, o código percorre as características reextraídas até a avaliação.


In [ ]:
metricas_dos_payloads = {}
for dataset_id, nivel in CENARIOS_PAYLOAD:
    conjunto = dados[dataset_id]
    vetores = caracteristicas.normalizar(conjunto.x)
    inferencia = inferir(modelo, vetores, LIMIAR_DECISAO)
    metricas_dos_payloads[dataset_id] = metricas_payload(
        tuple(int(valor) for valor in conjunto.y),
        tuple(int(valor) for valor in inferencia.rotulos),
        conjunto=dataset_id,
        nivel=nivel,
    )

avaliacao_payload = montar_avaliacao_payload(
    pipeline,
    metricas_dos_payloads,
)
problem_rows = []
for dataset_id, _ in CENARIOS_PAYLOAD:
    cenario = avaliacao_payload[dataset_id]
    metrics = cenario.metricas.to_dict()
    problem_rows.append({
        "Payload": metrics["nominal_factor"],
        "Fluxos DoS": metrics["total_dos"],
        "TP": metrics["tp"],
        "FN": metrics["fn"],
        "Recall": pct(metrics["recall"]),
        "FNR": pct(metrics["fnr"]),
    })

display(pd.DataFrame(problem_rows).style.hide(axis="index"))
display(Markdown("\n".join(
    f"- Payload **{row['Payload']}**: Recall **{row['Recall']}** e FNR **{row['FNR']}**."
    for row in problem_rows
)))


## 5. Registre três observações

Complete uma frase sobre cada ponto e execute a próxima célula.

**Pipeline NIDS:** TRÁFEGO/PCAP -> EXTRAÇÃO -> CARACTERÍSTICAS -> SCALER -> MLP -> DECISÃO -> [**AVALIAÇÃO**]

> **Foco desta seção:** Interprete e registre os resultados produzidos nas etapas anteriores.


In [ ]:
RESPOSTA = """
- **Ataques vetoriais:** [compare FGM minimal e PGD]
- **Payload:** [descreva o efeito observado nos três níveis]
- **Interpretação:** [registre o que os protocolos permitem concluir e o principal limite]
"""

display(Markdown(RESPOSTA))
